# PLOG

In [1]:
# --------------------------------------------------------
# Import packages
# --------------------------------------------------------
import matplotlib.pyplot as plt
import numpy as np
import jax.numpy as jnp

from diffPLOG2TROE.parametrization import Plog, Arrhenius

In [12]:
rate_constant = Plog(
    parameters=jnp.array([
        [0.01, 5.02e+21, -4.24, 898.9],
        [0.1, 5.31e+22, -4.24, 1184.0],
        [0.316, 1.38e+23, -4.22, 1376.0],
        [1.0, 3.09e+23, -4.17, 1621.0],
        [3.16, 5.45e+23, -4.09, 1911.0],
        [10.0, 6.35e+23, -3.97, 2222.0],
        [31.6, 3.68e+23, -3.75, 2501.0],
        [100.0, 7.29e+22, -3.41, 2660.0],
    ]),
    name="OH+NO=HONO"
)

# print(rate_constant.kinetic_constant(300, 0.215))
# print(rate_constant.kinetic_constant(300, 0.1))

print(rate_constant.kinetic_constant(300, 0.316))

arrhenius = Arrhenius(parameters=[1.38e+23, -4.22, 1376.0])
print(arrhenius.kinetic_constant(300))

ciao = jnp.array([
        [0.01, 5.02e+21, -4.24, 898.9],
        [0.1, 5.31e+22, -4.24, 1184.0],
        [0.316, 1.38e+23, -4.22, 1376.0],
        [1.0, 3.09e+23, -4.17, 1621.0],
        [3.16, 5.45e+23, -4.09, 1911.0],
        [10.0, 6.35e+23, -3.97, 2222.0],
        [31.6, 3.68e+23, -3.75, 2501.0],
        [100.0, 7.29e+22, -3.41, 2660.0],
    ])

# ciao = ciao.flatten()
print(ciao.shape)

483094601948.96423
483094601948.96423
(8, 4)


## Pressure Logarithmic Reaction
Introduced by J. Miller, as a generalized polynomial fitting for temperature and pressure dependent kinetic constants, by defining the following expression for the kinetic constant:
- Equation
  > $k_f \left(T, P_{i}\right) = \sum_{k=1}^{M} A_{i, k} \: T^{n_{i, k}} \: exp\left(-E_{act}^{i,k}/(RT)\right), \quad i=1, ..., Np, \quad M \geq 1$
  
  at a set of pressures, $P = P_{1}, P_{2}, ..., P_{Np}$. $M$ and $Np$ are user specified numbers. The extrapolation is bounded by the two pressure limits, $P_{1}$ and $P_{Np}$. To calculate $k \left(T, P_{i}\right)$ for any pressure, interpolate $logk$ as a linear function of $logP$. If $P$ is between $P_{i}$ and $P_{i+1}$ for any temperature a rate constant can be find from:
  > $logk_f \left(T, P\right) = logk_f\left(T, P_{i}\right) + \left(logP - logP_{i}\right) \frac{logk_f\left(T, P_{i+1}\right) - logk_f\left(T, P_{i}\right)}{logP_{i+1} - logP_{i}}$
- CHEMKIN representation
  >```
  >  NH3=NH2+H   .3497E+31  -5.224  111163.3
  >  PLOG / 0.1  .7230E+30  -5.316  110862.4 /
  >  PLOG / 1    .3497E+31  -5.224  111163.3 /
  >  PLOG / 10   .1975E+32  -5.155  111887.8 /
  >  PLOG / 100  .2689E+32  -4.920  112778.7 /
  >```
- Internal representation
    >```python
    >plog_constant = {
    >    "name": "NH3=NH2+H",
    >    "type": "plog",
    >    "rate-constant": {
    >        "coefficients": [
    >            [0.1, .7230E+30, -5.316, 110862.4],
    >            [1, .3497E+31, -5.224, 111163.3],
    >            [10, .1975E+32, -5.155, 111887.8],
    >            [100, .2689E+32, -4.920, 112778.7],
    >        ]
    >    }
    >}
    >```

In [ ]:
# --------------------------------------------------------
# Direct parameters initialization
# --------------------------------------------------------
rate_constant = Plog(
    parameters=jnp.array([
        [0.1, .7230E+30, -5.316, 110862.4],
        [1, .3497E+31, -5.224, 111163.3],
        [10, .1975E+32, -5.155, 111887.8],
        [100, .2689E+32, -4.920, 112778.7]
    ]),
    name="NH3=NH2+H",
)

In [ ]:
T_range = jnp.linspace(500, 3000, 300)
P_range = jnp.array([0.1, 1, 10, 100])
# P_range = jnp.logspace(jnp.log10(0.1), jnp.log10(100), 300)

k_plog = rate_constant.kinetic_constant(T_range, P_range)

plt.figure(figsize=(12, 8))

plt.subplot(1, 2, 1)
for i, j in enumerate(k_plog):
    plt.semilogy(T_range, j, label=f"P = {P_range[i]} [atm]")
plt.xlabel("Temperature [K]")
plt.ylabel("kinetic constant")
plt.legend()

T_range = jnp.array([500, 1000, 1500, 2000, 2500, 3000])
P_range = jnp.linspace(0.1, 100, 300)
k_plog = rate_constant.kinetic_constant(T_range, P_range)

plt.subplot(1, 2, 2)
for i in range(k_plog.shape[1]):
    plt.semilogy(P_range, k_plog[:, i], label=f"T = {T_range[i]} [K]")
plt.xlabel("Pressure [atm]")
plt.ylabel("kinetic constant")
plt.legend()
plt.show()